<a href="https://colab.research.google.com/github/Durvankur-Rajam/Impactsure_Project/blob/main/Week4_PDF_Extraction_Task13%2C14%2C15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pymupdf -q
print("PyMuPDF installed")

PyMuPDF installed


In [ ]:
!wget -q "https://arxiv.org/pdf/2109.07958" -O truthfulqa.pdf
!wget -q "https://arxiv.org/pdf/2303.08896" -O selfcheckgpt.pdf
!wget -q "https://arxiv.org/pdf/1706.03762" -O attention.pdf

print("PDFs downloaded:")
!ls -lh *.pdf

PDFs downloaded:
-rw-r--r-- 1 root root 2.2M Apr 12  2024 attention.pdf
-rw-r--r-- 1 root root 882K Oct 12  2023 selfcheckgpt.pdf
-rw-r--r-- 1 root root 801K Jan 23  2023 truthfulqa.pdf


In [ ]:
import fitz

def extract_text(pdf_path):
    doc = fitz.open(pdf_path)
    full_text = ""
    for page_num, page in enumerate(doc):
        text = page.get_text()
        full_text += f"\n--- Page {page_num + 1} ---\n{text}"
    doc.close()
    return full_text

pdfs = {
    "TruthfulQA": "truthfulqa.pdf",
    "SelfCheckGPT": "selfcheckgpt.pdf",
    "Attention": "attention.pdf"
}

extracted = {}
for name, path in pdfs.items():
    text = extract_text(path)
    extracted[name] = text
    print(f"{name}: {len(text)} characters extracted")

TruthfulQA: 92998 characters extracted
SelfCheckGPT: 53778 characters extracted
Attention: 39744 characters extracted


In [ ]:
for name, text in extracted.items():
    filename = f"{name.lower().replace(' ', '_')}_extracted.txt"
    with open(filename, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"Saved: {filename}")

print("\nAll text files saved")
!ls -lh *.txt

💾 Saved: truthfulqa_extracted.txt
💾 Saved: selfcheckgpt_extracted.txt
💾 Saved: attention_extracted.txt

All text files saved
-rw-r--r-- 1 root root 39K May  7 15:04 attention_extracted.txt
-rw-r--r-- 1 root root 53K May  7 15:04 selfcheckgpt_extracted.txt
-rw-r--r-- 1 root root 92K May  7 15:04 truthfulqa_extracted.txt


In [ ]:
for name, text in extracted.items():
    print(f"\n{'='*50}")
    print(f"{name} — First 500 chars:")
    print(f"{'='*50}")
    print(text[:500])


TruthfulQA — First 500 chars:

--- Page 1 ---
TruthfulQA: Measuring How Models Mimic Human Falsehoods
Stephanie Lin
University of Oxford
sylin07@gmail.com
Jacob Hilton
OpenAI
jhilton@openai.com
Owain Evans
University of Oxford
owaine@gmail.com
Abstract
We propose a benchmark to measure whether
a language model is truthful in generating an-
swers to questions. The benchmark comprises
817 questions that span 38 categories, includ-
ing health, law, ﬁnance and politics.
We
crafted questions that some humans would an-
swer falsel

SelfCheckGPT — First 500 chars:

--- Page 1 ---
SELFCHECKGPT: Zero-Resource Black-Box Hallucination Detection
for Generative Large Language Models
Potsawee Manakul, Adian Liusie, Mark J. F. Gales
ALTA Institute, Department of Engineering, University of Cambridge
pm574@cam.ac.uk, al826@cam.ac.uk, mjfg@eng.cam.ac.uk
Abstract
Generative Large Language Models (LLMs)
such as GPT-3 are capable of generating highly
fluent responses to a wide variety of user
prompts. How

In [ ]:
import re

def clean_text(raw_text):

    text = raw_text.replace('\x0c', ' ')
    text = text.replace('\u2019', "'")
    text = text.replace('\u201c', '"')
    text = text.replace('\u201d', '"')
    text = text.replace('\u2013', '-')
    text = text.replace('\u2014', '--')

    text = re.sub(r'--- Page \d+ ---', '', text)

    text = re.sub(r'http\S+', '', text)

    text = re.sub(r'\n{3,}', '\n\n', text)
    text = re.sub(r'[ \t]+', ' ', text)

    lines = text.split('\n')
    lines = [l for l in lines if not re.fullmatch(r'\s*\d+\s*', l)]

    lines = [l for l in lines if len(l.strip()) > 4 or l.strip() == '']

    text = '\n'.join(lines).strip()
    return text

In [ ]:
cleaned = {}
for name, raw in extracted.items():
    cleaned[name] = clean_text(raw)
    print(f"✅ {name}:")
    print(f"   Before: {len(raw):,} chars")
    print(f"   After:  {len(cleaned[name]):,} chars")
    print(f"   Reduced by: {100 - len(cleaned[name])*100//len(raw)}%\n")

✅ TruthfulQA:
   Before: 92,998 chars
   After:  90,212 chars
   Reduced by: 3%

✅ SelfCheckGPT:
   Before: 53,778 chars
   After:  51,906 chars
   Reduced by: 4%

✅ Attention:
   Before: 39,744 chars
   After:  38,135 chars
   Reduced by: 5%



In [ ]:
def split_paragraphs(text):

    paragraphs = text.split('\n\n')
    paragraphs = [p.strip() for p in paragraphs if len(p.strip()) > 50]
    return paragraphs

para_counts = {}
for name, text in cleaned.items():
    paras = split_paragraphs(text)
    para_counts[name] = paras
    print(f"📄 {name}: {len(paras)} paragraphs extracted")

📄 TruthfulQA: 46 paragraphs extracted
📄 SelfCheckGPT: 15 paragraphs extracted
📄 Attention: 15 paragraphs extracted


In [ ]:
for name, text in cleaned.items():
    filename = f"{name.lower().replace(' ', '_')}_clean.txt"
    with open(filename, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"💾 Saved: {filename}")

print("\nAll cleaned files saved ✅")
!ls -lh *_clean.txt

💾 Saved: truthfulqa_clean.txt
💾 Saved: selfcheckgpt_clean.txt
💾 Saved: attention_clean.txt

All cleaned files saved ✅
-rw-r--r-- 1 root root 38K May  7 15:06 attention_clean.txt
-rw-r--r-- 1 root root 51K May  7 15:06 selfcheckgpt_clean.txt
-rw-r--r-- 1 root root 89K May  7 15:06 truthfulqa_clean.txt


In [ ]:
sample = "TruthfulQA"
print("BEFORE cleaning:")
print(extracted[sample][:300])
print("\n" + "="*50)
print("AFTER cleaning:")
print(cleaned[sample][:300])

BEFORE cleaning:

--- Page 1 ---
TruthfulQA: Measuring How Models Mimic Human Falsehoods
Stephanie Lin
University of Oxford
sylin07@gmail.com
Jacob Hilton
OpenAI
jhilton@openai.com
Owain Evans
University of Oxford
owaine@gmail.com
Abstract
We propose a benchmark to measure whether
a language model is truthful in gen

AFTER cleaning:
TruthfulQA: Measuring How Models Mimic Human Falsehoods
Stephanie Lin
University of Oxford
sylin07@gmail.com
Jacob Hilton
OpenAI
jhilton@openai.com
Owain Evans
University of Oxford
owaine@gmail.com
Abstract
We propose a benchmark to measure whether
a language model is truthful in generating an-
swer


In [ ]:
!pip install transformers accelerate -q
print("Libraries ready")

Libraries ready


In [ ]:
from transformers import pipeline

# Load GPT-2 text generation pipeline
generator = pipeline("text-generation", model="gpt2", device=0)

# Pick a paragraph from TruthfulQA cleaned text
paras = para_counts["TruthfulQA"]

# Use paragraph 0 as context
context = paras[0]
question = "What is TruthfulQA and why was it created?"

print("📄 Context paragraph:")
print(context[:400])
print(f"\n❓ Question: {question}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


📄 Context paragraph:
TruthfulQA: Measuring How Models Mimic Human Falsehoods
Stephanie Lin
University of Oxford
sylin07@gmail.com
Jacob Hilton
OpenAI
jhilton@openai.com
Owain Evans
University of Oxford
owaine@gmail.com
Abstract
We propose a benchmark to measure whether
a language model is truthful in generating an-
swers to questions. The benchmark comprises
817 questions that span 38 categories, includ-
ing health, l

❓ Question: What is TruthfulQA and why was it created?


In [ ]:
def build_qa_prompt(context, question):
    prompt = f"""You are a helpful assistant. Use the context below to answer the question.

Context:
{context[:500]}

Question: {question}

Answer:"""
    return prompt

prompt = build_qa_prompt(context, question)
print("📝 Full Prompt:")
print(prompt)
print(f"\nPrompt length: {len(prompt)} characters")

📝 Full Prompt:
You are a helpful assistant. Use the context below to answer the question.

Context:
TruthfulQA: Measuring How Models Mimic Human Falsehoods
Stephanie Lin
University of Oxford
sylin07@gmail.com
Jacob Hilton
OpenAI
jhilton@openai.com
Owain Evans
University of Oxford
owaine@gmail.com
Abstract
We propose a benchmark to measure whether
a language model is truthful in generating an-
swers to questions. The benchmark comprises
817 questions that span 38 categories, includ-
ing health, law, ﬁnance and politics.
crafted questions that some humans would an-
swer falsely due to a false be

Question: What is TruthfulQA and why was it created?

Answer:

Prompt length: 648 characters


In [ ]:
output = generator(
    prompt,
    max_length=len(prompt.split()) + 100,
    num_return_sequences=1,
    temperature=0.7,
    do_sample=True,
    pad_token_id=generator.tokenizer.eos_token_id
)

full_output = output[0]['generated_text']
answer = full_output[len(prompt):]

print("Generated Answer:")
print(answer.strip())

Both `max_new_tokens` (=256) and `max_length`(=196) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated Answer:
It is not only true that humans are

falsely correct in their answers to the questions, but that they are false

if they do not ask questions.

Source: "A Comparison of True and False Beliefs About

Truthful Answers to Common Questions in Natural Language Processing."

Abstract:

A comparison of the two is a common,

common, and often asked question. The

factorial of truthfulness is a more general,

complex, and complex question, whereas a

complex question is not at all easy to answer. The

question of truthfulness is easy for humans to answer,

though difficult for other species.

Source: "A Comparison of False and True Beliefs About

Truthful Answers to Common Questions in Natural

Language Processing."

Abstract:

The following sample question is a common

question, with a simple answer and a simple

question for the answer: "Isn't it true that in any

factorial of the two, you know that you are

true if you are truthful?

Source: "A Comparison of False and True


In [ ]:
questions = {
    "TruthfulQA": "What is TruthfulQA and why was it created?",
    "SelfCheckGPT": "How does SelfCheckGPT detect hallucinations?",
    "Attention": "What is the attention mechanism in transformers?"
}

print("="*60)
for name, question in questions.items():
    context = para_counts[name][0]
    prompt = build_qa_prompt(context, question)

    output = generator(
        prompt,
        max_length=len(prompt.split()) + 80,
        num_return_sequences=1,
        temperature=0.7,
        do_sample=True,
        pad_token_id=generator.tokenizer.eos_token_id
    )

    full_output = output[0]['generated_text']
    answer = full_output[len(prompt):].strip()

    print(f"\n PDF: {name}")
    print(f"❓ Q: {question}")
    print(f"A: {answer[:200]}")
    print("="*60)

Both `max_new_tokens` (=256) and `max_length`(=176) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Both `max_new_tokens` (=256) and `max_length`(=166) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 PDF: TruthfulQA
❓ Q: What is TruthfulQA and why was it created?
A: It is not clear whether this is an accurate measure of the

true nature of falsehoods, but it is possible that it is useful to

analyze these questions.

Example: We asked a young man whether he belie


Both `max_new_tokens` (=256) and `max_length`(=168) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



 PDF: SelfCheckGPT
❓ Q: How does SelfCheckGPT detect hallucinations?
A: ual,
and grammatical, choices difficult. A well-

understood way to reduce this problem is to generate

a set of standard-size LLMs in a highly

fluent way, and then run them on a

big-endian (in the 

 PDF: Attention
❓ Q: What is the attention mechanism in transformers?
A: al.gomez@utexas.edu
Simon J. Jahn∗†
University of Toronto
simon@utexas.edu
Aidan N. Gomez∗†
University of Toronto
al.gomez@utexas.edu
Simon J. Jahn∗†
University of Toronto
simon@utexas.edu
John W. Loh
